# Persistent tensor-parallel generation with Transformers + Accelerate

Select the **SPMD Python** kernel and set **Processes: 2** before running. Transformers owns the native tensor-parallel plan; Accelerate supplies convenient distributed process/device state. We load the model once, inspect local TP state, generate, change the prompt, and generate again from the same resident model.

> This example targets two CUDA GPUs, downloads an ungated model, and has not been GPU-validated in this repository. Native TP APIs and supported model families can change between Transformers releases.

In [ ]:
%pip install -q "transformers>=5,<6" "accelerate>=1,<2" safetensors sentencepiece

The demo-only dependencies are isolated in the first code cell. If the environment changed, restart the distributed kernel group once, keep **Processes: 2**, and continue. Each later cell assumes the model stays alive on every rank.

In [ ]:
import os
from datetime import timedelta

import torch
from accelerate import PartialState

rank = int(os.environ["RANK"])
world_size = int(os.environ["WORLD_SIZE"])
assert world_size >= 2, "Set Processes to 2 and restart the kernel group"
assert torch.cuda.device_count() >= world_size, "This demo needs one CUDA GPU per local rank"
state = PartialState(timeout=timedelta(hours=24))
torch.cuda.set_device(state.local_process_index)
print({"rank": state.process_index, "world_size": state.num_processes, "device": str(state.device)})

Load an ungated Qwen checkpoint with Transformers native tensor parallelism. All ranks execute this cell together; `tp_plan="auto"` uses the already-running distributed world rather than launching a separate job. The 0.5B checkpoint keeps the download and memory footprint modest for a first run.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype=torch.bfloat16,
    tp_plan="auto",
)
model.eval()
print(f"rank {rank}: loaded {model_id} on {state.device}")

Inspect the plan and one local parameter shard before inference. The exact plan representation is model- and Transformers-version-dependent, so the cell uses public attributes when present and degrades to a useful local shape summary.

In [ ]:
tp_plan = getattr(model, "_tp_plan", getattr(model.config, "base_model_tp_plan", None))
name, parameter = next((n, p) for n, p in model.named_parameters() if "proj" in n)
local = parameter.to_local() if hasattr(parameter, "to_local") else parameter
print({
    "rank": rank,
    "tp_plan": tp_plan,
    "parameter": name,
    "global_shape": tuple(parameter.shape),
    "local_shape": tuple(local.shape),
    "placements": tuple(getattr(parameter, "placements", ())),
})

Generation remains SPMD: every rank enters the model call so tensor-parallel collectives have peers. Only rank 0 prints the user-facing text, while each rank prints a small diagnostic visible in its output tab.

In [ ]:
prompt = "Explain tensor parallelism in one concise sentence."
messages = [{"role": "user", "content": prompt}]
text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(text, return_tensors="pt").to(state.device)
with torch.inference_mode():
    generated = model.generate(**inputs, max_new_tokens=48, do_sample=False)
new_tokens = generated[:, inputs.input_ids.shape[1]:]
print(f"rank {rank}: local generation result shape={tuple(generated.shape)}")
if state.is_main_process:
    first_answer = tokenizer.decode(new_tokens[0], skip_special_tokens=True)
    print(first_answer)

The model and tokenizer are still resident. Change only the prompt and generate again; no subprocess or model reload is hidden in this cell.

In [ ]:
prompt = "Now give a two-line pseudocode example of an all-reduce."
messages = [{"role": "user", "content": prompt}]
text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(text, return_tensors="pt").to(state.device)
with torch.inference_mode():
    generated = model.generate(**inputs, max_new_tokens=64, do_sample=False)
new_tokens = generated[:, inputs.input_ids.shape[1]:]
print(f"rank {rank}: reused model id={id(model)}, local tokens={new_tokens.numel()}")
if state.is_main_process:
    second_answer = tokenizer.decode(new_tokens[0], skip_special_tokens=True)
    print(second_answer)